# SMILES 2026 Method 1 Colab Runner

This notebook updates the GitHub repo in a Drive-backed workspace, installs dependencies, and runs the SAPLMA-style MLP baseline plus the requested ablation study on `dataset.csv` with `Qwen/Qwen2.5-0.5B`.

Default layer selection uses `layer_rankings.csv`, which currently resolves to `layer 15`.

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

In [ ]:
import os
import subprocess
from pathlib import Path

TARGET_FOLDER = Path('/content/drive/MyDrive/hallucination_detection')
REPO_URL = 'https://github.com/olgafilimonova2004/hallucination_detection_draft.git'
REPO_NAME = 'hallucination_detection_draft'
REPO_PATH = TARGET_FOLDER / REPO_NAME
AUTO_STASH = True

TARGET_FOLDER.mkdir(parents=True, exist_ok=True)

def run(cmd: str, cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    print(f'$ {cmd}')
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd is not None else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f'command failed with exit code {result.returncode}: {cmd}')
    return result

print('TARGET_FOLDER =', TARGET_FOLDER)
print('REPO_PATH =', REPO_PATH)

In [ ]:
if not REPO_PATH.exists():
    run(f'git clone {REPO_URL}', cwd=TARGET_FOLDER)

run('git remote -v', cwd=REPO_PATH)
run('git branch --show-current', cwd=REPO_PATH)
run('git fetch origin', cwd=REPO_PATH)
status = run('git status --short', cwd=REPO_PATH, check=False).stdout.strip()

if status:
    print('Local changes detected.')
    if AUTO_STASH:
        run('git stash push -u -m "colab-auto-stash"', cwd=REPO_PATH)
    else:
        raise RuntimeError('Repo is dirty. Set AUTO_STASH = True or clean it manually.')

run('git pull --ff-only origin main', cwd=REPO_PATH)
run('git log --oneline -1', cwd=REPO_PATH)

os.chdir(REPO_PATH)
print('cwd =', Path.cwd())

In [ ]:
run('pip install -q -r requirements.txt', cwd=REPO_PATH)

In [ ]:
import torch

print('torch.cuda.is_available() =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU =', torch.cuda.get_device_name(0))
else:
    print('GPU not available. Switch Colab runtime to GPU.')

## Method 1 runs

The default baseline is:

- token mode: `response_last`
- layers: `auto`
- `auto-top-k = 1`

Given the current `layer_rankings.csv`, that means **layer 15**.

The ablation sweep below compares MLP depth `4/3/2`, dropout `0.0/0.3`, and regularization `off/on` for combined `L1 + L2`.

In [ ]:
run(
    'python method1_saplma/run_method1.py '
    '--subset 80 '
    '--layers auto '
    '--auto-top-k 1 '
    '--token-mode response_last '
    '--batch-size 2 '
    '--cache-dtype float16',
    cwd=REPO_PATH,
)

In [ ]:
run(
    'python method1_saplma/run_method1.py '
    '--layers auto '
    '--auto-top-k 1 '
    '--token-mode response_last '
    '--batch-size 2 '
    '--cache-dtype float16',
    cwd=REPO_PATH,
)

In [ ]:
for layer in [12, 14, 15, 16]:
    run(
        'python method1_saplma/run_method1.py '
        f'--layers {layer} '
        '--token-mode response_last '
        '--batch-size 2 '
        '--cache-dtype float16 '
        f'--output-file method1_saplma/artifacts/method1_layer_{layer}.json',
        cwd=REPO_PATH,
    )

In [ ]:
for token_mode in ['last_token', 'response_last', 'response_mean']:
    run(
        'python method1_saplma/run_method1.py '
        '--layers 15 '
        f'--token-mode {token_mode} '
        '--batch-size 2 '
        '--cache-dtype float16 '
        f'--output-file method1_saplma/artifacts/method1_{token_mode}.json',
        cwd=REPO_PATH,
    )

## Regularization and Depth Ablation

This sweep reuses the cached hidden states and ranks the configurations by **mean validation AUROC**. It writes a leaderboard plus `best_config.json` under `method1_saplma/artifacts/ablation/`.

In [ ]:
run(
    'python method1_saplma/run_ablation.py '
    '--layers auto '
    '--auto-top-k 1 '
    '--token-mode response_last '
    '--batch-size 2 '
    '--cache-dtype float16',
    cwd=REPO_PATH,
)

In [ ]:
import json
import pandas as pd
from pathlib import Path

ablation_dir = REPO_PATH / 'method1_saplma' / 'artifacts' / 'ablation'
leaderboard = pd.read_csv(ablation_dir / 'ablation_results.csv')
display(leaderboard.sort_values(['mean_val_auroc', 'mean_val_f1'], ascending=[False, False]).head(12))

best_config = json.loads((ablation_dir / 'best_config.json').read_text())
print(json.dumps(best_config, indent=2))

In [ ]:
import json

ablation_dir = REPO_PATH / 'method1_saplma' / 'artifacts' / 'ablation'
best_payload = json.loads((ablation_dir / 'best_config.json').read_text())
best = best_payload['best_config']
layers_arg = ','.join(str(layer) for layer in best_payload['layers'])
hidden_dims_arg = ','.join(str(width) for width in best['hidden_dims_list'])

run(
    'python method1_saplma/run_method1.py '
    f'--layers {layers_arg} '
    f"--token-mode {best_payload['token_mode']} "
    f'--hidden-dims {hidden_dims_arg} '
    f"--dropout-p {best['dropout_p']} "
    f"--l1-lambda {best['l1_lambda']} "
    f"--l2-weight-decay {best['l2_weight_decay']} "
    '--batch-size 2 '
    '--cache-dtype float16 '
    '--output-file method1_saplma/artifacts/method1_best_from_ablation.json',
    cwd=REPO_PATH,
)